Загрузить очищенную выборку. Разбить её на тестовую и обучающую в пропорции **25%–75%**.

Деление — стандартным `train_test_split` из sklearn.


In [143]:
import os
import mlflow

from sklearn.model_selection import train_test_split
import pandas as pd
import numpy

from sklearn.preprocessing import StandardScaler, OrdinalEncoder, OneHotEncoder

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression

from sklearn.metrics import mean_absolute_percentage_error, mean_absolute_error, mean_squared_error


In [144]:
from pathlib import Path

def find_project_root() -> Path:
    here = Path.cwd().resolve()
    for candidate in [here, *here.parents]:
        if (candidate / "data" / "clean_data.pkl").exists() or (candidate / "data" / "dataset.csv").exists():
            return candidate
    raise FileNotFoundError(
        "Не найден data/clean_data.pkl. Сначала выполните eda/eda.ipynb "
        f"(текущая директория: {here})"
    )

PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / "data"

df = pd.read_pickle(DATA_DIR / "clean_data.pkl")
df = df.rename(columns={"Selling_Price": "target"})
X_train, X_test, y_train, y_test = train_test_split(
    df.drop("target", axis=1), df["target"], test_size=0.25, random_state=2
)
print("PROJECT_ROOT:", PROJECT_ROOT)
print("train/test:", X_train.shape, X_test.shape)


PROJECT_ROOT: /Users/gleb/Desktop/иис/iis
train/test: (224, 9) (75, 9)


Создать переменные, содержащие названия столбцов с **числовыми** и **категориальными** признаками.


In [145]:
cat_features = X_train.select_dtypes(include=["category", "object"]).columns.to_list()
num_features = X_train.select_dtypes(include=["number"]).columns.to_list()
print("cat_features:", cat_features)
print("num_features:", num_features)


cat_features: ['Car_Name', 'Fuel_Type', 'Selling_type', 'Transmission', 'Owner', 'Driven_run']
num_features: ['Year', 'Present_Price', 'Driven_kms']


Создать pipeline обработки признаков и обучения модели:
- для числовых признаков — `StandardScaler`;
- для категориальных — `TargetEncoder` (классификация) или `OrdinalEncoder` (регрессия);
- в качестве модели — `RandomForest`.

Pipeline состоит из двух шагов: `transform` и `regression`.

In [146]:
s_scaler = StandardScaler()
l_encoder = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=99999999)
regressor = RandomForestRegressor()

preprocessor = ColumnTransformer(
    transformers=[
        ("num", s_scaler, num_features),
        ("cat", l_encoder, cat_features),
    ],
    remainder="drop",
)


Обучить **baseline-модель** и получить метрики качества на тестовой выборке.

Для регрессии: **mae, mape, mse**.

Baseline — простая модель без особенных настроек, отправная точка для сравнения.

In [147]:
pipeline = Pipeline(steps=[("preprocessor", preprocessor), ("model", regressor)])
pipeline.fit(X_train, y_train)


Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num', StandardScaler(),
                                                  ['Year', 'Present_Price',
                                                   'Driven_kms']),
                                                 ('cat',
                                                  OrdinalEncoder(handle_unknown='use_encoded_value',
                                                                 unknown_value=99999999),
                                                  ['Car_Name', 'Fuel_Type',
                                                   'Selling_type',
                                                   'Transmission', 'Owner',
                                                   'Driven_run'])])),
                ('model', RandomForestRegressor())])

In [148]:
predictions = pipeline.predict(X_test)

metrics = {}
metrics["mae"] = mean_absolute_error(y_test, predictions)
metrics["mape"] = mean_absolute_percentage_error(y_test, predictions)
metrics["mse"] = mean_squared_error(y_test, predictions)

metrics


{'mae': 0.5652933333333336,
 'mape': 0.29679924275820857,
 'mse': 1.1545981426666667}

Залогировать baseline-модель в новый эксперимент. Зарегистрировать модель.

Подготовить:
- сигнатуру модели (`infer_signature`);
- пример входных данных;
- файл `requirements.txt`.

Логировать со flavour **sklearn**: `mlflow.sklearn.log_model(...)`.


In [149]:
TRACKING_SERVER_HOST = "127.0.0.1"
TRACKING_SERVER_PORT = 5001

registry_uri = f"http://{TRACKING_SERVER_HOST}:{TRACKING_SERVER_PORT}"
tracking_uri = f"http://{TRACKING_SERVER_HOST}:{TRACKING_SERVER_PORT}"

mlflow.set_tracking_uri(tracking_uri)
mlflow.set_registry_uri(registry_uri)


In [150]:
EXPERIMENT_NAME = "estate_project"
RUN_NAME = "baseline model"
REGISTRY_MODEL_NAME = "estate_model_rf"


In [151]:
from mlflow.models import infer_signature

input_example = X_train.head(5)
signature = infer_signature(model_input=X_train.head(5))


/Users/gleb/Desktop/иис/iis/.venv/lib/python3.11/site-packages/mlflow/types/utils.py:407: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(


In [152]:
req_file = str(PROJECT_ROOT / "requirements.txt")
art = str(PROJECT_ROOT / "comment.txt")
params_dict = pipeline.get_params()


In [153]:
exp = mlflow.get_experiment_by_name(EXPERIMENT_NAME)
experiment_id = exp.experiment_id if exp is not None else mlflow.create_experiment(EXPERIMENT_NAME)

with mlflow.start_run(run_name=RUN_NAME, experiment_id=experiment_id) as run:
    run_id = run.info.run_id
    mlflow.sklearn.log_model(
        pipeline,
        artifact_path="models",
        signature=signature,
        input_example=input_example,
        pip_requirements=req_file,
    )
    mlflow.log_metrics(metrics)
    mlflow.log_artifact(art)
    mlflow.log_params(params_dict)

run = mlflow.get_run(run_id)
assert run.info.status == "FINISHED"
print("baseline run_id:", run_id)


2026/09/29 11:58:49 INFO mlflow.tracking._tracking_service.client: 🏃 View run baseline model at: http://127.0.0.1:5001/#/experiments/1/runs/c79d3cfd971c4417b128016cb405c4c9.
2026/09/29 11:58:49 INFO mlflow.tracking._tracking_service.client: 🧪 View experiment at: http://127.0.0.1:5001/#/experiments/1.


baseline run_id: c79d3cfd971c4417b128016cb405c4c9


С использованием библиотеки **sklearn** создать дополнительные признаки, обучить модель, залогировать её.

Для 2–3 числовых признаков — `PolynomialFeatures(degree=2)` (с последующим шкалированием).  
Если нет временных рядов — `KBinsDiscretizer` / аналоги; у нас дополнительно `QuantileTransformer` и `SplineTransformer`.

1. `X_train_fe_sklearn = X_train.copy()`
2. `ColumnTransformer` = baseline-трансформы + новые
3. `fit_transform` → расширенный датафрейм
4. сохранить названия столбцов (при необходимости логировать)
5. pipeline: ColumnTransformer → модель → predict → log в MLflow (`RUN_NAME` понятный)


In [155]:
from sklearn.preprocessing import QuantileTransformer, SplineTransformer, PolynomialFeatures

X_train_sklearn = X_train.copy()

pf = PolynomialFeatures(degree=2)
qt = QuantileTransformer(n_quantiles=100)
sp = SplineTransformer(n_knots=3, degree=3)

pf_pipeline = Pipeline(steps=[
    ("poly", pf),
    ("scale", StandardScaler()),
])

preprocessor_sklearn = ColumnTransformer(
    transformers=[
        ("num", s_scaler, num_features),
        ("cat", l_encoder, cat_features),
        ("quantile", qt, num_features),
        ("poly", pf_pipeline, ["Year", "Driven_kms"]),
        ("spline", sp, ["Driven_kms"]),
    ],
    remainder="drop",
)


In [156]:
X_train_sklearn_raw = preprocessor_sklearn.fit_transform(X_train_sklearn)
X_train_sklearn = pd.DataFrame(
    X_train_sklearn_raw,
    columns=preprocessor_sklearn.get_feature_names_out(),
)

# сохраняем названия столбцов после FE (для артефакта MLflow)
fe_cols_file = "fe_sklearn_columns.txt"
with open(fe_cols_file, "w") as f:
    f.write("\n".join(X_train_sklearn.columns.tolist()))

with pd.option_context("display.max_rows", 5, "display.max_columns", None):
    display(X_train_sklearn)


,num__Year,num__Present_Price,num__Driven_kms,cat__Car_Name,cat__Fuel_Type,cat__Selling_type,cat__Transmission,cat__Owner,cat__Driven_run,quantile__Year,quantile__Present_Price,quantile__Driven_kms,poly__1,poly__Year,poly__Driven_kms,poly__Year^2,poly__Year Driven_kms,poly__Driven_kms^2,spline__Driven_kms_sp_0,spline__Driven_kms_sp_1,spline__Driven_kms_sp_2,spline__Driven_kms_sp_3,spline__Driven_kms_sp_4
0,0.115848,-0.787328,-0.551107,15.0,2.0,1.0,1.0,0.0,1.0,0.449495,0.115025,0.217172,0.0,0.115848,-0.551107,0.115257,-0.551880,-0.175831,0.140503,0.663690,0.195779,2.823558e-05,0.0
1,0.466521,0.738487,0.125527,56.0,1.0,0.0,1.0,0.0,0.0,0.621212,0.898990,0.690802,0.0,0.466521,0.125527,0.466403,0.127270,-0.078449,0.095385,0.640298,0.263502,8.151886e-04,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
222,0.817195,0.302633,0.112841,58.0,1.0,0.0,1.0,0.0,0.0,0.803030,0.775558,0.680380,0.0,0.817195,0.112841,0.817724,0.115048,-0.081149,0.096126,0.640930,0.262159,7.846722e-04,0.0
223,-0.585500,-0.782642,-0.776375,44.0,2.0,1.0,1.0,0.0,1.0,0.222222,0.154401,0.056743,0.0,-0.585500,-0.776375,-0.586513,-0.777944,-0.187185,0.158194,0.666372,0.175433,8.542102e-07,0.0


In [157]:
pipeline_sklearn = Pipeline(steps=[
    ("transform", preprocessor_sklearn),
    ("model", regressor),
])

model_sklearn = pipeline_sklearn.fit(X_train, y_train)

predictions = model_sklearn.predict(X_test)
metrics = {}
metrics["mae"] = mean_absolute_error(y_test, predictions)
metrics["mape"] = mean_absolute_percentage_error(y_test, predictions)
metrics["mse"] = mean_squared_error(y_test, predictions)
metrics


{'mae': 0.5731186666666669,
 'mape': 0.32084791492876485,
 'mse': 1.0917871830666668}

In [158]:
experiment_id = mlflow.get_experiment_by_name(EXPERIMENT_NAME).experiment_id
RUN_NAME = "fe_sklearn"

with mlflow.start_run(run_name=RUN_NAME, experiment_id=experiment_id) as run:
    run_id = run.info.run_id
    mlflow.sklearn.log_model(
        model_sklearn,
        artifact_path="models",
        signature=signature,
        input_example=input_example,
        pip_requirements=req_file,
    )
    mlflow.log_metrics(metrics)
    mlflow.log_artifact(fe_cols_file)
    mlflow.log_artifact(art)
    mlflow.log_params(model_sklearn.get_params())

run = mlflow.get_run(run_id)
assert run.info.status == "FINISHED"
print("fe_sklearn run_id:", run_id)
metrics


2026/09/29 11:59:27 INFO mlflow.tracking._tracking_service.client: 🏃 View run fe_sklearn at: http://127.0.0.1:5001/#/experiments/1/runs/7cd1563708634b5290f3a03170f5a4ad.
2026/09/29 11:59:27 INFO mlflow.tracking._tracking_service.client: 🧪 View experiment at: http://127.0.0.1:5001/#/experiments/1.


fe_sklearn run_id: 7cd1563708634b5290f3a03170f5a4ad


{'mae': 0.5731186666666669,
 'mape': 0.32084791492876485,
 'mse': 1.0917871830666668}

Отобрать **N** наиболее важных признаков.

Отбор можно выполнить:
- **экспертно** (по результатам EDA) — основной путь;

При экспертном отборе:
1. Продумать, сколько и каких признаков оставить на основе EDA.
2. Сохранить названия и индексы в два файла (залогировать с моделью).
3. Добавить в пайплайн из п.10 шаг отбора столбцов по индексам.
4. Обучить, предсказать, залогировать (`RUN_NAME` понятный).


In [159]:
feature_names = list(X_train_sklearn.columns)
priority_keys = [
    "Present_Price", "Year", "Driven_kms", "Driven_run",
    "Fuel_Type", "Selling_type", "Transmission", "Car_Name",
]
preferred = [c for c in feature_names if any(k in c for k in priority_keys)]

n_select = max(3, int(round(len(feature_names) * 0.4)))  # ~40% из диапазона 20–70%
if len(preferred) >= n_select:
    selected_cols = preferred[:n_select]
else:
    rest = [c for c in feature_names if c not in preferred]
    selected_cols = preferred + rest[: n_select - len(preferred)]

selected_idx = [feature_names.index(c) for c in selected_cols]
print("Всего признаков после FE:", len(feature_names))
print("Отобрано N =", len(selected_cols))
print("Столбцы:", selected_cols)
print("Индексы:", selected_idx)


Всего признаков после FE: 23
Отобрано N = 9
Столбцы: ['num__Year', 'num__Present_Price', 'num__Driven_kms', 'cat__Car_Name', 'cat__Fuel_Type', 'cat__Selling_type', 'cat__Transmission', 'cat__Driven_run', 'quantile__Year']
Индексы: [0, 1, 2, 3, 4, 5, 6, 8, 9]


In [ ]:
selected_features = X_train_sklearn.loc[:, selected_cols]
selected_features.head()

,num__Year,num__Present_Price,num__Driven_kms,cat__Car_Name,cat__Fuel_Type,cat__Selling_type,cat__Transmission,cat__Driven_run,quantile__Year
0,0.115848,-0.787328,-0.551107,15.0,2.0,1.0,1.0,1.0,0.449495
1,0.466521,0.738487,0.125527,56.0,1.0,0.0,1.0,0.0,0.621212
2,-0.585500,-0.133220,0.329383,66.0,1.0,0.0,1.0,0.0,0.222222
3,-0.936173,-0.329354,-0.432545,73.0,2.0,0.0,1.0,2.0,0.156566
4,0.466521,-0.230198,-0.563793,49.0,2.0,0.0,1.0,1.0,0.621212


In [161]:
class ColumnExtractor(object):
    def __init__(self, cols):
        self.cols = cols

    def transform(self, X):
        return X[:, self.cols]

    def fit(self, X, y=None):
        return self


In [162]:
with open("selected_skl_idx.txt", "w+") as f:
    f.write(str(selected_idx))
with open("selected_skl_cols.txt", "w+") as f:
    f.write(str(selected_cols))


In [163]:
fs_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor_sklearn),
    ("feature_selector", ColumnExtractor(selected_idx)),
    ("model", regressor),
])

fs_pipeline.fit(X_train, y_train)

predictions_fs = fs_pipeline.predict(X_test)
metrics = {}
metrics["mae"] = mean_absolute_error(y_test, predictions_fs)
metrics["mape"] = mean_absolute_percentage_error(y_test, predictions_fs)
metrics["mse"] = mean_squared_error(y_test, predictions_fs)

experiment_id = mlflow.get_experiment_by_name(EXPERIMENT_NAME).experiment_id
RUN_NAME = "expert_feature_selection"

with mlflow.start_run(run_name=RUN_NAME, experiment_id=experiment_id) as run:
    run_id = run.info.run_id
    mlflow.sklearn.log_model(
        fs_pipeline,
        artifact_path="models",
        signature=signature,
        input_example=input_example,
        pip_requirements=req_file,
    )
    mlflow.log_metrics(metrics)
    mlflow.log_artifact("selected_skl_cols.txt")
    mlflow.log_artifact("selected_skl_idx.txt")
    mlflow.log_params(fs_pipeline.get_params())

run = mlflow.get_run(run_id)
assert run.info.status == "FINISHED"
print("expert_feature_selection run_id:", run_id)
metrics


2026/09/29 11:59:49 INFO mlflow.tracking._tracking_service.client: 🏃 View run expert_feature_selection at: http://127.0.0.1:5001/#/experiments/1/runs/e1a0d62221334a58acc54b9a24d9c0c0.
2026/09/29 11:59:49 INFO mlflow.tracking._tracking_service.client: 🧪 View experiment at: http://127.0.0.1:5001/#/experiments/1.


expert_feature_selection run_id: e1a0d62221334a58acc54b9a24d9c0c0


{'mae': 0.519998666666667,
 'mape': 0.31628970527656447,
 'mse': 1.0800047036000004}

С помощью **Optuna** настроить оптимальные параметры для модели, показывающей лучший результат.

Для Random Forest настраивать:
- количество деревьев;
- глубину дерева;
- `max_features` в интервале от 0.1 до 1.0.

Провести **не менее 10 trials**. Для регрессии оптимизировать **MAE**, явно указав minimize/maximize.

Залогировать модель, зарегистрировать очередную версию.

In [ ]:
import optuna

def objective(trial):
    depth = trial.suggest_int("depth", 1, 10)
    max_features = trial.suggest_float("max_features", 0.1, 1.0)
    n_estimators = trial.suggest_int("n_estimators", 1, 3)

    opt_pipeline = Pipeline(steps=[
        ("preprocessor", preprocessor_sklearn),
        ("feature_selector", ColumnExtractor(selected_idx)),
        ("model", RandomForestRegressor(
            n_estimators=n_estimators,
            max_depth=depth,
            max_features=max_features,
            random_state=2,
        )),
    ])
    opt_pipeline.fit(X_train, y_train)
    preds = opt_pipeline.predict(X_test)
    return mean_absolute_error(y_test, preds)

study = optuna.create_study(direction="minimize")
study.optimize(objective, n_trials=10)

print("Number of finished trials:", len(study.trials))
print("Best trial:", study.best_trial.params)
print("Best MAE:", study.best_value)


[I 2026-09-29 11:59:56,541] A new study created in memory with name: no-name-1966c4cc-92cd-4a4b-a4e5-7b0a852338a9
[I 2026-09-29 11:59:56,565] Trial 0 finished with value: 0.9638499259343081 and parameters: {'depth': 3, 'max_features': 0.545787609929211, 'n_estimators': 2}. Best is trial 0 with value: 0.9638499259343081.
[I 2026-09-29 11:59:56,577] Trial 1 finished with value: 1.258511111111111 and parameters: {'depth': 9, 'max_features': 0.25679802044984507, 'n_estimators': 1}. Best is trial 0 with value: 0.9638499259343081.
[I 2026-09-29 11:59:56,589] Trial 2 finished with value: 0.6575346092796092 and parameters: {'depth': 7, 'max_features': 0.7647788292996128, 'n_estimators': 2}. Best is trial 2 with value: 0.6575346092796092.
[I 2026-09-29 11:59:56,598] Trial 3 finished with value: 1.7282330660997547 and parameters: {'depth': 2, 'max_features': 0.6830562941079192, 'n_estimators': 2}. Best is trial 2 with value: 0.6575346092796092.
[I 2026-09-29 11:59:56,608] Trial 4 finished with v

Number of finished trials: 10
Best trial: {'depth': 9, 'max_features': 0.5947508521263627, 'n_estimators': 1}
Best MAE: 0.6401246031746032


In [165]:
best_params = study.best_trial.params
opt_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor_sklearn),
    ("feature_selector", ColumnExtractor(selected_idx)),
    ("model", RandomForestRegressor(
        max_depth=best_params["depth"],
        max_features=best_params["max_features"],
        n_estimators=best_params["n_estimators"],
        random_state=2,
    )),
])

opt_pipeline.fit(X_train, y_train)
predictions = opt_pipeline.predict(X_test)
metrics = {}
metrics["mae"] = mean_absolute_error(y_test, predictions)
metrics["mape"] = mean_absolute_percentage_error(y_test, predictions)
metrics["mse"] = mean_squared_error(y_test, predictions)

experiment_id = mlflow.get_experiment_by_name(EXPERIMENT_NAME).experiment_id
RUN_NAME = "optuna"

with mlflow.start_run(run_name=RUN_NAME, experiment_id=experiment_id) as run:
    run_id = run.info.run_id
    mlflow.sklearn.log_model(
        opt_pipeline,
        artifact_path="models",
        signature=signature,
        input_example=input_example,
        pip_requirements=req_file,
        registered_model_name=REGISTRY_MODEL_NAME,
    )
    mlflow.log_metrics(metrics)
    mlflow.log_artifact(art)
    mlflow.log_params(opt_pipeline.get_params())

run = mlflow.get_run(run_id)
assert run.info.status == "FINISHED"
print("optuna run_id:", run_id)
metrics


Registered model 'estate_model_rf' already exists. Creating a new version of this model...
2026/09/29 12:00:15 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: estate_model_rf, version 4
Created version '4' of model 'estate_model_rf'.
2026/09/29 12:00:15 INFO mlflow.tracking._tracking_service.client: 🏃 View run optuna at: http://127.0.0.1:5001/#/experiments/1/runs/6383c9e05c0c4e42ba6ff164ac6caa84.
2026/09/29 12:00:15 INFO mlflow.tracking._tracking_service.client: 🧪 View experiment at: http://127.0.0.1:5001/#/experiments/1.


optuna run_id: 6383c9e05c0c4e42ba6ff164ac6caa84


{'mae': 0.6401246031746032,
 'mape': 0.2508407635528844,
 'mse': 1.0942869845677714}

Проанализировать все прогоны и выбрать модель с наилучшим качеством.  
Обучить эту модель на **всей выборке** (не только train). Метрики качества мерить уже не обязательно, но залогировать:
- сигнатуру;
- пример входных данных;
- `requirements.txt`;
- список используемых столбцов.

Зарегистрировать версию в реестре и установить тег / alias **Production**.

### Как выбирали лучшую модель

Сравнивали runs в MLflow по **MAE на test** (меньше — лучше):
- `baseline model`
- `fe_sklearn`
- `expert_feature_selection`
- `optuna`

В качестве Production-конфигурации берём **`pipeline_sklearn` (FE без отбора)** — полный набор сгенерированных признаков sklearn; её обучаем на всей выборке.


In [166]:
df_full = pd.read_pickle(DATA_DIR / "clean_data.pkl")
df_full = df_full.rename(columns={"Selling_Price": "target"})
Y = df_full["target"]
X = df_full.drop("target", axis=1)
print("Full sample:", X.shape)

Full sample: (299, 9)


In [167]:
production_model = fs_pipeline.fit(X, Y)

In [168]:
experiment_id = mlflow.get_experiment_by_name(EXPERIMENT_NAME).experiment_id
RUN_NAME = "best_selection"

input_example = X.head(5)
signature = infer_signature(model_input=X.head(5))

cols_file = "production_columns.txt"
with open(cols_file, "w") as f:
    f.write("\n".join(X.columns.tolist()))

with mlflow.start_run(run_name=RUN_NAME, experiment_id=experiment_id) as run:
    run_id = run.info.run_id
    mlflow.sklearn.log_model(
        production_model,
        artifact_path="models",
        signature=signature,
        input_example=input_example,
        pip_requirements=req_file,
        registered_model_name=REGISTRY_MODEL_NAME,
    )
    mlflow.log_artifact(art)
    mlflow.log_artifact(cols_file)
    mlflow.log_params(production_model.get_params())

run = mlflow.get_run(run_id)
assert run.info.status == "FINISHED"

client = mlflow.tracking.MlflowClient()
versions = client.search_model_versions(f"name='{REGISTRY_MODEL_NAME}'")
latest = sorted(versions, key=lambda v: int(v.version))[-1]
client.set_registered_model_alias(REGISTRY_MODEL_NAME, "Production", latest.version)
client.set_model_version_tag(REGISTRY_MODEL_NAME, latest.version, "stage", "Production")
print("Production version:", latest.version)
print("Production run_id:", run_id)
run


/Users/gleb/Desktop/иис/iis/.venv/lib/python3.11/site-packages/mlflow/types/utils.py:407: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
Registered model 'estate_model_rf' already exists. Creating a new version of this model...
2026/09/29 12:07:23 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. M

Production version: 5
Production run_id: 57f68ed8c2b2447386d1427fc5459e96


<Run: data=<RunData: metrics={}, params={'feature_selector': '<__main__.ColumnExtractor object at 0x1350e6150>',
 'memory': 'None',
 'model': 'RandomForestRegressor()',
 'model__bootstrap': 'True',
 'model__ccp_alpha': '0.0',
 'model__criterion': 'squared_error',
 'model__max_depth': 'None',
 'model__max_features': '1.0',
 'model__max_leaf_nodes': 'None',
 'model__max_samples': 'None',
 'model__min_impurity_decrease': '0.0',
 'model__min_samples_leaf': '1',
 'model__min_samples_split': '2',
 'model__min_weight_fraction_leaf': '0.0',
 'model__monotonic_cst': 'None',
 'model__n_estimators': '100',
 'model__n_jobs': 'None',
 'model__oob_score': 'False',
 'model__random_state': 'None',
 'model__verbose': '0',
 'model__warm_start': 'False',
 'preprocessor': "ColumnTransformer(transformers=[('num', StandardScaler(),\n"
                 "                                 ['Year', 'Present_Price', "
                 "'Driven_kms']),\n"
                 "                                ('cat',\n